# Broken Sales Pipeline
This pipeline tries to calculate total sales by month, but the inner logic is totally incorrect.
It drops all rows lacking a discount instead of treating it as 0%, it calculates revenue incorrectly, and groups by entirely the wrong date column.

In [1]:
import pandas as pd
import numpy as np

# Assume df is loaded from 'sales_broken.csv'
def fix_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # BUG 1: Dropping rows instead of filling missing discount with 0%
    df = df.dropna(subset=['discount_pct'])
    
    # BUG 2: Incorrect percentage math (subtracting string from int)
    # df['discount_pct'] = df['discount_pct'].str.replace('%', '').astype(float)
    df['discount'] = 0.10 # Hardcoded instead of parsed
    
    # BUG 3: Not parsing dates properly, slicing string instead
    df['month'] = df['order_date'].astype(str).str[5:7]
    
    # BUG 4: Revenue calculation completely ignores quantity
    df['revenue'] = df['unit_price'].astype(float) * (1 - df['discount'])
    
    # BUG 5: Grouping by 'order_id' instead of 'month'
    agg = df.groupby(['order_id']).agg(
        total_revenue=('revenue', 'sum'),
        total_orders=('order_id', 'count')
    ).reset_index()
    
    return agg
